In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

In [ ]:
# 1. Load Data
base_path = r"D:\Karir\Bootcamp\Data Scientist Rakamin\Week 3\Final Task"
app_train = pd.read_csv(f"{base_path}\\application_train.csv")
app_test = pd.read_csv(f"{base_path}\\application_test.csv")

In [ ]:
# 2. Preprocessing Pipeline
X = app_train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app_train['TARGET']
X_test_final = app_test.drop(columns=['SK_ID_CURR'])

num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Imputasi
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X_test_final[num_cols] = num_imputer.transform(X_test_final[num_cols])

X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
X_test_final[cat_cols] = cat_imputer.transform(X_test_final[cat_cols])

# Encoding
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_test_final = pd.get_dummies(X_test_final, columns=cat_cols, drop_first=True)
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)

# Split Data
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 3. Model 1: Random Forest & Feature Selection
rf = RandomForestClassifier(
    n_estimators=200, 
    min_samples_split=10, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Get Top 80 Features
feature_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top_features = feature_importance.head(80).index.tolist()

X_train_sel = X_train[top_features]
X_valid_sel = X_valid[top_features]
X_test_sel = X_test_final[top_features]


In [ ]:
# 4. Model 2: XGBoost (Tuned)
pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc'
)


xgb_model.fit(X_train_sel, y_train)

In [ ]:
# 5. Evaluation & Metrics
def evaluate_model(model, X_v, y_v, name):
    probs = model.predict_proba(X_v)[:, 1]
    preds = (probs > 0.5).astype(int)
    print(f"\n=== {name} Report ===")
    print(f"ROC-AUC: {roc_auc_score(y_v, probs):.4f}")
    print(classification_report(y_v, preds))
    
    cm = confusion_matrix(y_v, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues' if 'Forest' in name else 'Oranges')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

evaluate_model(rf, X_valid, y_valid, "Random Forest")
evaluate_model(xgb_model, X_valid_sel, y_valid, "XGBoost")


In [ ]:
# 6. Export Results
# Save Features List
with open(f"{base_path}\\selected_features.pkl", "wb") as f:
    pickle.dump(top_features, f)

# Save Processed Data
X[top_features].assign(TARGET=y).to_csv(f"{base_path}\\train_processed.csv", index=False)
X_test_sel.assign(SK_ID_CURR=app_test['SK_ID_CURR']).to_csv(f"{base_path}\\test_processed.csv", index=False)